# Section 10 — Evaluation: Flat Baseline vs Hierarchical Retrieval


## Objective

This notebook evaluates the usefulness of the generated multi-resolution
hierarchy for scientific knowledge retrieval.

The evaluation compares two retrieval strategies:


### 1. Flat retrieval baseline

The baseline searches directly over all original TKH method entities.

The pipeline is:


Question

↓

Question embedding

↓

All method nodes

↓

Top-k retrieved methods



This represents a conventional retrieval approach without using hierarchy.



### 2. Hierarchical retrieval

The proposed approach searches the generated Level-3 labelled hierarchy.

The pipeline is:


Question

↓

Question embedding

↓

Level-3 hierarchy concepts

↓

Top-k retrieved concepts



The objective is to investigate whether semantic abstraction improves retrieval
quality and produces more meaningful scientific concepts.



## Evaluation design


Both approaches use:

- the same question embeddings;
- the same retrieval method;
- the same top-k value;
- the same evaluation metrics.


The evaluation includes:


### Quantitative evaluation

- Precision
- Recall
- F1 score


### Qualitative evaluation

For each benchmark question, we compare:

- original expected methods;
- flat retrieval results;
- hierarchical retrieval results.


Because the benchmark frequently uses abbreviated method names
(e.g., MACE, GAP, DeepH) while TKH contains expanded descriptions, exact
string matching is strict.

Therefore, qualitative semantic analysis is also included.

## Clone repository

In [1]:
from pathlib import Path

REPO_DIR = Path(
    "/content/tkh-hierarchy-project"
)


if not REPO_DIR.exists():

    !git clone https://github.com/mohamadghoroobi/tkh-hierarchy-project.git


%cd /content/tkh-hierarchy-project

Cloning into 'tkh-hierarchy-project'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 178 (delta 106), reused 112 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (178/178), 17.80 MiB | 20.55 MiB/s, done.
Resolving deltas: 100% (106/106), done.
/content/tkh-hierarchy-project


In [2]:
!ls

artifacts     README.md
data	      retrieval_evaluation_results.csv
deliverables  snapshot_metadata.json
LICENSE       t1_statistics.json
notebooks     TKH_Multi_Resolution_Semantic_Abstraction_Report.pdf


## Imports

In [3]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

from collections import defaultdict

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## Define paths

In [4]:
PROJECT_DIR = Path(
    "/content/tkh-hierarchy-project"
)


DATA_DIR = (
    PROJECT_DIR
    /
    "data"
)


LABELLED_DIR = (
    PROJECT_DIR
    /
    "artifacts"
    /
    "hierarchy"
    /
    "labelled"
)


TKH_PATH = (
    DATA_DIR
    /
    "tkh_collection10.json"
)


QUESTIONS_PATH = (
    DATA_DIR
    /
    "questions.csv"
)


GROUND_TRUTH_PATH = (
    DATA_DIR
    /
    "ground_truth.json"
)


print(TKH_PATH)
print(QUESTIONS_PATH)
print(GROUND_TRUTH_PATH)
print(LABELLED_DIR)

/content/tkh-hierarchy-project/data/tkh_collection10.json
/content/tkh-hierarchy-project/data/questions.csv
/content/tkh-hierarchy-project/data/ground_truth.json
/content/tkh-hierarchy-project/artifacts/hierarchy/labelled


## Load TKH graph

In [5]:
with open(
    TKH_PATH,
    "r",
    encoding="utf-8"
) as f:

    tkh = json.load(f)


nodes = tkh["nodes"]

hyperedges = tkh["hyperedges"]


node_lookup = {

    node["id"]:
        node

    for node in nodes

}


print(
    "Nodes:",
    len(nodes)
)


print(
    "Hyperedges:",
    len(hyperedges)
)

Nodes: 5798
Hyperedges: 1429


## Load questions

In [6]:
questions_df = pd.read_csv(
    QUESTIONS_PATH,
    sep=";"
)


print(
    questions_df.shape
)


questions_df.head()

(18, 3)


,question_id,question,type
0,Q1,Which methods (by Feb 2026) are best suited fo...,A
1,Q2,Which methods (by Feb 2026) are best suited fo...,A
2,Q3,Which methods (by Feb 2026) are best suited fo...,A
3,Q4,Which methods (by Feb 2026) are best suited fo...,A
4,Q5,Which methods (by Feb 2026) are best suited fo...,A


## Load ground truth

In [7]:
with open(
    GROUND_TRUTH_PATH,
    "r",
    encoding="utf-8"
) as f:

    ground_truth = json.load(f)


print(
    "Questions in ground truth:",
    len(ground_truth)
)

Questions in ground truth: 18


## Load labelled hierarchy

In [8]:
labelled_hierarchies = {}


for year in [2020,2022,2024,2026]:

    path = (
        LABELLED_DIR /
        f"hierarchy_{year}_labelled.json"
    )


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        labelled_hierarchies[year] = json.load(f)


print(
    labelled_hierarchies.keys()
)

dict_keys([2020, 2022, 2024, 2026])


## Load embedding model

In [9]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Flat Retrieval Baseline

The flat baseline ignores the hierarchy.

It directly searches original TKH method entities.

This provides a reference point to understand whether hierarchical abstraction
improves retrieval.

## Build flat candidates

In [10]:
flat_units = []


for node_id,node in node_lookup.items():

    if node.get("type") != "method":
        continue


    label = node.get(
        "surface_form"
    )


    if label is None:
        continue


    flat_units.append({

        "persistent_id":
            node_id,

        "label":
            label,

        "type":
            "method",

        "text":
            "Method: " + label

    })


flat_df = pd.DataFrame(
    flat_units
)


print(
    "Flat method candidates:",
    len(flat_df)
)


flat_df.head()

Flat method candidates: 448


,persistent_id,label,type,text
0,meth_00001,Generative Adversarial Networks,method,Method: Generative Adversarial Networks
1,meth_00002,Atomic Simulation Environment,method,Method: Atomic Simulation Environment
2,meth_00003,CLIP,method,Method: CLIP
3,meth_00004,CatBoost,method,Method: CatBoost
4,meth_00005,Graph Convolutional Network,method,Method: Graph Convolutional Network


## Encode flat candidates

In [11]:
flat_embeddings = model.encode(

    flat_df["text"].tolist(),

    normalize_embeddings=True

)


print(
    flat_embeddings.shape
)

(448, 384)


## Flat retrieval function

In [12]:
def retrieve_flat_top_k(
    embedding,
    k=5
):

    scores = cosine_similarity(

        embedding.reshape(1,-1),

        flat_embeddings

    )[0]


    indices = np.argsort(
        scores
    )[::-1][:k]


    results=[]


    for idx in indices:

        row = flat_df.iloc[idx]


        results.append({

            "persistent_id":
                row["persistent_id"],

            "label":
                row["label"],

            "score":
                float(scores[idx])

        })


    return results

## Encode questions

In [13]:
question_embeddings = model.encode(

    questions_df["question"].tolist(),

    normalize_embeddings=True

)


print(
    question_embeddings.shape
)

(18, 384)


## Run flat retrieval

In [14]:
flat_results=[]


for idx,embedding in enumerate(
    question_embeddings
):

    flat_results.append({

        "question_id":
            questions_df.iloc[idx]["question_id"],

        "retrieved":
            retrieve_flat_top_k(
                embedding,
                k=5
            )

    })


print(
    len(flat_results)
)

18


# Hierarchical Retrieval

This experiment evaluates retrieval using the generated hierarchy.

Instead of searching all original method entities, the system searches
Level-3 labelled hierarchy concepts.

Level-3 is selected because it contains specific scientific methods while
still benefiting from the abstraction process.

## Build Level-3 hierarchy retrieval units

In [15]:
hierarchy_units = []


for year, hierarchy in labelled_hierarchies.items():

    level = "3"


    for record in hierarchy["levels"][level]:

        label = record.get(
            "label"
        )


        if label is None:
            continue


        gloss = (
            record.get("gloss")
            or ""
        )


        text = (
            str(label)
            +
            ". "
            +
            str(gloss)
        )


        hierarchy_units.append({

            "year":
                year,

            "persistent_id":
                record["persistent_id"],

            "label":
                label,

            "text":
                text

        })


hierarchy_df = pd.DataFrame(
    hierarchy_units
)


print(
    "Hierarchy candidates:",
    len(hierarchy_df)
)


hierarchy_df.head()

Hierarchy candidates: 13631


,year,persistent_id,label,text
0,2020,arti_00001,Neural Message Passing for Quantum Chemistry,Neural Message Passing for Quantum Chemistry.
1,2020,arti_00002,SchNet: A continuous-filter convolutional neur...,SchNet: A continuous-filter convolutional neur...
2,2020,arti_00016,Deep Potential Molecular Dynamics: a scalable ...,Deep Potential Molecular Dynamics: a scalable ...
3,2020,arti_00020,Crystal Graph Convolutional Neural Networks fo...,Crystal Graph Convolutional Neural Networks fo...
4,2020,arti_00021,Graph Networks as a Universal Machine Learning...,Graph Networks as a Universal Machine Learning...


## Encode hierarchy concepts

In [16]:
hierarchy_embeddings = model.encode(

    hierarchy_df["text"].tolist(),

    normalize_embeddings=True

)


print(
    hierarchy_embeddings.shape
)

(13631, 384)


## Hierarchical retrieval function

In [17]:
def retrieve_hierarchical_top_k(
    embedding,
    k=5
):

    scores = cosine_similarity(

        embedding.reshape(1,-1),

        hierarchy_embeddings

    )[0]


    indices = np.argsort(
        scores
    )[::-1][:k]


    results = []


    for idx in indices:

        row = hierarchy_df.iloc[idx]


        results.append({

            "persistent_id":
                row["persistent_id"],

            "label":
                row["label"],

            "score":
                float(scores[idx])

        })


    return results

## Run hierarchical retrieval

In [18]:
hierarchical_results = []


for idx, embedding in enumerate(
    question_embeddings
):

    hierarchical_results.append({

        "question_id":
            questions_df.iloc[idx]["question_id"],

        "retrieved":
            retrieve_hierarchical_top_k(
                embedding,
                k=5
            )

    })


print(
    len(hierarchical_results)
)

18


## # Retrieval Evaluation

Both retrieval approaches use the same evaluation protocol.

A prediction is considered correct when the normalized retrieved label exactly
matches a normalized gold method.

Because scientific names may have different surface forms, qualitative
inspection is also performed.

## Gold extraction

In [19]:
def normalize_text(
    text
):

    return (
        str(text)
        .lower()
        .replace(
            "-",
            ""
        )
        .replace(
            " ",
            ""
        )
        .replace(
            "_",
            ""
        )
    )

In [20]:
def extract_gold_methods(
    qid
):

    item = ground_truth[qid]


    return [

        normalize_text(x)

        for x in item.get(
            "expected_methods",
            []
        )

    ]

## Evaluate retrieval

In [21]:
def evaluate_retrieval(
    retrieval_results
):

    rows=[]


    for result in retrieval_results:

        qid = result["question_id"]


        gold = set(
            extract_gold_methods(qid)
        )


        predicted = set(

            normalize_text(
                r["label"]
            )

            for r in result["retrieved"]

        )


        hits = (
            gold &
            predicted
        )


        precision = (

            len(hits)
            /
            max(
                len(predicted),
                1
            )

        )


        recall = (

            len(hits)
            /
            max(
                len(gold),
                1
            )

        )


        if precision + recall > 0:

            f1 = (
                2 *
                precision *
                recall
                /
                (
                    precision +
                    recall
                )
            )

        else:

            f1 = 0



        rows.append({

            "question_id":
                qid,

            "gold_methods":
                list(gold),

            "predicted_methods":
                list(predicted),

            "hits":
                list(hits),

            "precision":
                precision,

            "recall":
                recall,

            "f1":
                f1

        })


    return pd.DataFrame(rows)

## Evaluate both approaches

In [22]:
flat_eval_df = evaluate_retrieval(
    flat_results
)


hierarchical_eval_df = evaluate_retrieval(
    hierarchical_results
)


flat_eval_df.head()

,question_id,gold_methods,predicted_methods,hits,precision,recall,f1
0,Q1,"[m3gnet, gap, chgnet, ace, mace, equiformerv2,...",[universalmachinelearninginteratomicpotentials...,[],0.0,0.0,0
1,Q2,"[hamgnn, d4ft, deeph]","[virtualcrystalapproximation, universalmodelsf...",[],0.0,0.0,0
2,Q3,"[deephe3, xdeeph]","[deeplearningdfthamiltonian, vanderwaalsdensit...",[],0.0,0.0,0
3,Q4,"[hessiantraining, vgnn, macef]","[frozenphononmethod, linearresponsephononmetho...",[],0.0,0.0,0
4,Q5,"[gnn+gpp, beadmapping]","[diffusivemoleculardynamics, non‑equilibriummo...",[],0.0,0.0,0


## Overall metrics

# Retrieval Comparison Analysis


The following table compares the retrieved concepts from the flat baseline and
the hierarchical approach.


The purpose is not only to measure exact matching performance, but also to
understand the type of scientific concepts returned by each method.


The flat baseline searches the original method nodes and may return:

- broad scientific concepts;
- general machine learning terms;
- descriptions from individual papers.


The hierarchical approach searches Level-3 abstract concepts.

Because Level-3 entities are already semantically grouped, retrieved concepts
are expected to be closer to the scientific method categories represented in
the benchmark.

In [23]:
flat_metrics = {

    "method":
        "Flat retrieval",

    "precision":
        flat_eval_df["precision"].mean(),

    "recall":
        flat_eval_df["recall"].mean(),

    "f1":
        flat_eval_df["f1"].mean()

}


hierarchical_metrics = {

    "method":
        "Hierarchical retrieval",

    "precision":
        hierarchical_eval_df["precision"].mean(),

    "recall":
        hierarchical_eval_df["recall"].mean(),

    "f1":
        hierarchical_eval_df["f1"].mean()

}


comparison_df = pd.DataFrame([

    flat_metrics,

    hierarchical_metrics

])


comparison_df

,method,precision,recall,f1
0,Flat retrieval,0.0,0.0,0.0
1,Hierarchical retrieval,0.0,0.0,0.0


## Final comparison table

In [24]:
comparison_df.style.format({

    "precision":
        "{:.3f}",

    "recall":
        "{:.3f}",

    "f1":
        "{:.3f}"

})

,method,precision,recall,f1
0,Flat retrieval,0.000,0.000,0.000
1,Hierarchical retrieval,0.000,0.000,0.000


## Qualitative comparison

In [25]:
for i in range(5):

    print("="*80)

    print(
        "QUESTION:",
        questions_df.iloc[i]["question_id"]
    )


    print("\nFLAT:")

    for r in flat_results[i]["retrieved"]:

        print(
            "-",
            r["label"]
        )


    print("\nHIERARCHICAL:")

    for r in hierarchical_results[i]["retrieved"]:

        print(
            "-",
            r["label"]
        )

QUESTION: Q1

FLAT:
- Machine-learned interatomic potentials
- Machine Learning Interatomic Potentials
- Transformer-based Inter-Atomic Potentials
- universal interatomic potentials
- Universal machine learning interatomic potentials

HIERARCHICAL:
- Benchmarking of interatomic potentials
- Benchmarking of interatomic potentials
- The Gaussian approximation potential (GAP) enables atomistic simulations with close-to-DFT accuracy at computational costs many orders of magnitude lower than quantum-mechanical methods.
- The Gaussian approximation potential (GAP) enables atomistic simulations with close-to-DFT accuracy at computational costs many orders of magnitude lower than quantum-mechanical methods.
- The Gaussian approximation potential (GAP) enables atomistic simulations with close-to-DFT accuracy at computational costs many orders of magnitude lower than quantum-mechanical methods.
QUESTION: Q2

FLAT:
- Large‑scale Atomic/Molecular Massively Parallel Simulator
- Modified Embedded At

## Save outputs

In [26]:
OUTPUT_DIR = (
    PROJECT_DIR /
    "deliverables" /
    "evaluation"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [27]:
flat_eval_df.to_csv(

    OUTPUT_DIR /
    "flat_retrieval_results.csv",

    index=False

)


hierarchical_eval_df.to_csv(

    OUTPUT_DIR /
    "hierarchical_retrieval_results.csv",

    index=False

)


comparison_df.to_csv(

    OUTPUT_DIR /
    "retrieval_comparison.csv",

    index=False

)


print(
    "Saved evaluation files."
)

Saved evaluation files.


# Summary

This notebook evaluated the usefulness of the TKH hierarchy for scientific
retrieval.

Two approaches were compared:

## 1. Flat retrieval baseline

The baseline searches all original method nodes directly.

## 2. Hierarchical retrieval

The proposed approach searches Level-3 labelled hierarchy concepts.

Both approaches use the same question embeddings, top-k retrieval procedure,
and evaluation metrics.

The comparison provides an experimental analysis of whether the generated
multi-resolution hierarchy improves downstream retrieval.

A limitation remains that benchmark methods often use abbreviations while TKH
labels contain expanded scientific descriptions. Therefore, exact matching
metrics underestimate semantic similarity.

Future improvements should include semantic entity linking before final
benchmark comparison.

## Git Push

In [28]:
from pathlib import Path

CLEAN = Path("/content/drive/MyDrive/Apply/Germany/ConstructorLabs/10_evaluation_flat_vs_hierarchical_retrieval.ipynb")
REPO_FILE = Path(
    "/content/tkh-hierarchy-project/"
    "notebooks/10_evaluation_flat_vs_hierarchical_retrieval.ipynb"
)

print("Clean file exists:", CLEAN.exists())
print("Repo file exists :", REPO_FILE.exists())

Clean file exists: False
Repo file exists : False


In [ ]:
import shutil

shutil.copy2(CLEAN, REPO_FILE)

print("Clean notebook copied into repository.")

Clean notebook copied into repository.


In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/10_extrinsic_retrieval.ipynb
	retrieval_evaluation_results.csv

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
!git add -A

In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   notebooks/10_extrinsic_retrieval.ipynb
	new file:   retrieval_evaluation_results.csv



In [ ]:
!git config --global user.name "mohamadghoroobi"
!git config --global user.email "m.ghoroobi@gmail.com"

In [ ]:
commit_message = """ feat(evaluation): compare flat baseline and hierarchical retrieval over TKH


Evaluate the practical value of the constructed temporal multi-resolution
hierarchy through comparative downstream retrieval experiments.


- implement flat retrieval baseline over original TKH method entities

- implement hierarchical retrieval over Level-3 labelled hierarchy concepts

- use the same question embeddings and top-k retrieval setting for fair comparison

- load benchmark questions and ground truth method annotations

- construct retrieval candidates from TKH method nodes and hierarchy clusters

- generate semantic embeddings for retrieval candidates and benchmark questions

- perform cosine similarity based top-k retrieval

- evaluate both retrieval strategies using precision, recall, and F1 metrics

- create comparative evaluation tables between flat and hierarchical retrieval

- provide qualitative analysis of retrieved concepts for benchmark questions

- document vocabulary mismatch between abbreviated benchmark methods and
  expanded TKH entity descriptions

- export retrieval results and comparison metrics for final report analysis

"""

with open("/tmp/commit_message.txt", "w", encoding="utf-8") as f:
    f.write(commit_message)

In [ ]:
!git commit -F /tmp/commit_message.txt

[main cc48dbf]  feat(evaluation): implement extrinsic retrieval evaluation over TKH hierarchy
 2 files changed, 20 insertions(+)
 create mode 100644 notebooks/10_extrinsic_retrieval.ipynb
 create mode 100644 retrieval_evaluation_results.csv


In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

assert token, "GITHUB_TOKEN not found"
print("Token loaded successfully")

Token loaded successfully


In [ ]:
import os
import subprocess
from pathlib import Path

username = "mohamadghoroobi"

env = os.environ.copy()

env["GITHUB_USER"] = "mohamadghoroobi"
env["GITHUB_TOKEN"] = token
env["GIT_TERMINAL_PROMPT"] = "0"


askpass = Path("/tmp/git_askpass.sh")

askpass.write_text(
"""#!/bin/sh
case "$1" in
  *Username*) echo "$GITHUB_USER" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""
)

askpass.chmod(0o700)

env["GIT_ASKPASS"] = str(askpass)

print("Git authentication prepared")

Git authentication prepared


In [ ]:
subprocess.run(
    ["git", "push", "origin", "main"],
    env=env,
    check=True
)

print("Push completed successfully")

In [ ]:
!git status